In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [2]:
!wget https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
!unzip car+evaluation.zip

--2025-04-02 16:19:54--  https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘car+evaluation.zip’

car+evaluation.zip      [ <=>                ]   6.19K  --.-KB/s    in 0s      

2025-04-02 16:19:54 (89.9 MB/s) - ‘car+evaluation.zip’ saved [6342]

Archive:  car+evaluation.zip
  inflating: car.c45-names           
  inflating: car.data                
  inflating: car.names               


In [3]:
import pandas as pd

df = pd.read_csv('car.data', sep=',', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'])
print(len(df))
df

1728


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [4]:
df.columns

Index(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'], dtype='object')

In [5]:
df['class'].unique()

array(['unacc', 'acc', 'vgood', 'good'], dtype=object)

In [6]:
import numpy as np
import torch

def concatenate_text(x):
    
    if x['buying'] == 'vhigh':
        b = 'very high'
    elif x['buying'] == 'high':
        b = 'high'
    elif x['buying'] == 'med':
        b = 'medium'
    elif x['buying'] == 'low':
        b = 'low'

    if x['maint'] == 'vhigh':
        m = 'very high'
    elif x['maint'] == 'high':
        m = 'high'
    elif x['maint'] == 'med':
        m = 'medium'
    elif x['maint'] == 'low':
        m = 'low'

    if x['doors'] == '5more':
        d = '5 or more'
    else:
        d = x['doors']

    if x['persons'] == 'more':
        p = 'more than 4'
    else:
        p = x['persons']

    if x['lug_boot'] == 'med':
        l = 'medium'
    else:
        l = x['lug_boot']

    if x['safety'] == 'med':
        s = 'medium'
    else:
        s = x['safety']

    
    text = "".join([f"I have information about a car. ",
            f"The buying price of it is {b}. ",
            f"The maintenance price of it is {m}. ",
            f"It has {d} doors. ",
            f"It has places for {p} people. ",
            f"The size of luggage boot is {l}. ",
            f"The safety of the car is estimated as {s}."])

    
    return text

concatenate_text(df.iloc[0])

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 2 doors. It has places for 2 people. The size of luggage boot is small. The safety of the car is estimated as low.'

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['label'] = y_train
X_test['label'] = y_test

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

<ipython-input-7-3d6e14cd6762>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
<ipython-input-7-3d6e14cd6762>:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


In [8]:
X_train['text'].iloc[0]

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for more than 4 people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [9]:
len(X_train)

1382

In [10]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.6 MB/s eta 0:00:00


In [11]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import evaluate

# Define label mappings
# id2label = {0: "NOT-DONATE", 1: "DONATE"}
# label2id = {"NOT-DONATE": 0, "DONATE": 1}

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

def tokenize_function(examples):
    # Adjust based on the structure of your dataset
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

In [12]:
tokenized_train_dataset[0].keys()

dict_keys(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'label', 'text', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [13]:
tokenized_train_dataset[0]['text']

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for more than 4 people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [14]:
tokenized_train_dataset[0]['label']

0

In [38]:
from transformers import BertForSequenceClassification
from transformers import DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, auc
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import sklearn.metrics as metrics

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('cuda')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Roc Auc
1,1.054500,1.041128,0.504349
2,0.934000,1.020522,0.667343
3,0.974200,1.060233,0.588940
4,0.908000,0.864901,0.763898
5,0.861500,0.910052,0.780784
6,0.839200,0.962401,0.774807
7,0.817300,1.042894,0.758929
8,0.835400,0.863768,0.764300
9,0.792400,0.851196,0.769555
10,0.923800,0.839465,0.771460


{'eval_loss': 0.8394649028778076, 'eval_roc_auc': 0.7714600687461299, 'eval_runtime': 1.2951, 'eval_samples_per_second': 267.155, 'eval_steps_per_second': 4.633, 'epoch': 10.0}
test f1 0.5494314168316535
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.5836756847693877
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [40]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# # Evaluation metric
# auc = evaluate.load("roc_auc")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
#     return auc.compute(prediction_scores=predictions, references=labels)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.098600,0.190667,0.996813
2,0.070500,0.126188,0.997824
3,0.079300,0.145393,0.997341
4,0.084600,0.125127,0.998545
5,0.061300,0.137857,0.998290
6,0.057900,0.138979,0.996051
7,0.064100,0.125619,0.997759
8,0.065600,0.157103,0.997009
9,0.045600,0.147961,0.997906
10,0.067700,0.142653,0.998372


{'eval_loss': 0.07420004159212112, 'eval_roc_auc': 0.9993648865539484, 'eval_runtime': 1.3144, 'eval_samples_per_second': 263.23, 'eval_steps_per_second': 4.565, 'epoch': 20.0}
test f1 0.9773691081099514
test precision 0.9796257616726843
test recall 0.976878612716763
test accuracy 0.976878612716763


train f1 0.9891092642752296
train precision 0.989429574016413
train recall 0.9891461649782923
train accuracy 0.9891461649782923
